In [1]:
import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn
import joblib
import warnings
from pathlib import Path
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, classification_report
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier
warnings.filterwarnings('ignore')

In [2]:
url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
df = pd.read_csv(url)

print(f"Shape : {df.shape}")
print(f"\nColonnes : {list(df.columns)}")
print(f"\nDistribution churn :")
print(df['Churn'].value_counts())
print(f"\nTaux de churn : {df['Churn'].value_counts(normalize=True)['Yes']:.1%}")
df.head()

Shape : (7043, 21)

Colonnes : ['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']

Distribution churn :
Churn
No     5174
Yes    1869
Name: count, dtype: int64

Taux de churn : 26.5%


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [3]:
# TotalCharges contient des espaces vides → convertir en numérique
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Supprimer customerID (inutile pour la prédiction)
df = df.drop(columns=['customerID'])

# Valeurs manquantes (11 lignes dans TotalCharges)
print(f"Valeurs manquantes avant : {df.isnull().sum().sum()}")
df = df.dropna()
print(f"Valeurs manquantes après : {df.isnull().sum().sum()}")
print(f"Shape finale : {df.shape}")

# Encoder la cible
df['Churn'] = (df['Churn'] == 'Yes').astype(int)
print(f"\nDistribution finale :")
print(df['Churn'].value_counts())

Valeurs manquantes avant : 11
Valeurs manquantes après : 0
Shape finale : (7032, 20)

Distribution finale :
Churn
0    5163
1    1869
Name: count, dtype: int64


In [4]:
# Colonnes catégorielles
cat_cols = df.select_dtypes(include='object').columns.tolist()
print(f"Colonnes catégorielles ({len(cat_cols)}) : {cat_cols}")

# Label encoding pour les binaires, get_dummies pour les autres
binary_cols = [c for c in cat_cols if df[c].nunique() == 2]
multi_cols = [c for c in cat_cols if df[c].nunique() > 2]

# Encoder les binaires
le = LabelEncoder()
for col in binary_cols:
    df[col] = le.fit_transform(df[col])

# One-hot encoding pour les multi-classes
df = pd.get_dummies(df, columns=multi_cols, drop_first=True)

print(f"\nShape après encodage : {df.shape}")
print(f"Colonnes finales : {list(df.columns)}")

Colonnes catégorielles (15) : ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']

Shape après encodage : (7032, 31)
Colonnes finales : ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'PaperlessBilling', 'MonthlyCharges', 'TotalCharges', 'Churn', 'MultipleLines_No phone service', 'MultipleLines_Yes', 'InternetService_Fiber optic', 'InternetService_No', 'OnlineSecurity_No internet service', 'OnlineSecurity_Yes', 'OnlineBackup_No internet service', 'OnlineBackup_Yes', 'DeviceProtection_No internet service', 'DeviceProtection_Yes', 'TechSupport_No internet service', 'TechSupport_Yes', 'StreamingTV_No internet service', 'StreamingTV_Yes', 'StreamingMovies_No internet service', 'StreamingMovies_Yes', 'Contract_One year', 'Contract_Two year', 'PaymentMethod_Credit card (automa

In [5]:
# Features métier pertinentes pour le churn télécom
df['charges_per_month_tenure'] = df['MonthlyCharges'] / (df['tenure'] + 1)
df['total_charges_ratio'] = df['TotalCharges'] / (df['MonthlyCharges'] * (df['tenure'] + 1) + 1)
df['is_new_client'] = (df['tenure'] <= 6).astype(int)
df['is_loyal_client'] = (df['tenure'] >= 48).astype(int)
df['high_spender'] = (df['MonthlyCharges'] >= 70).astype(int)

print("Nouvelles features ajoutées :")
new_feats = ['charges_per_month_tenure', 'total_charges_ratio',
             'is_new_client', 'is_loyal_client', 'high_spender']
for f in new_feats:
    print(f"  - {f}")

print(f"\nShape finale : {df.shape}")

Nouvelles features ajoutées :
  - charges_per_month_tenure
  - total_charges_ratio
  - is_new_client
  - is_loyal_client
  - high_spender

Shape finale : (7032, 36)


In [6]:
X = df.drop(columns=['Churn'])
y = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

param_grids = {
    "LogisticRegression": {
        "model": [LogisticRegression(max_iter=2000, random_state=42)],
        "model__C": [0.01, 0.1, 1, 10],
        "model__penalty": ["l1", "l2"],
        "model__solver": ["liblinear"]
    },
    "RandomForest": {
        "model": [RandomForestClassifier(random_state=42, class_weight='balanced')],
        "model__n_estimators": [100, 200, 300],
        "model__max_depth": [3, 5, 7, None],
        "model__min_samples_split": [2, 5, 10]
    },
    "XGBoost": {
        "model": [XGBClassifier(random_state=42, eval_metric='logloss',
                                scale_pos_weight=y_train.value_counts()[0]/y_train.value_counts()[1])],
        "model__n_estimators": [100, 200, 300],
        "model__max_depth": [3, 4, 5],
        "model__learning_rate": [0.05, 0.1, 0.2],
        "model__subsample": [0.8, 1.0]
    }
}

best_models = {}
results = {}
mlflow.set_experiment("churn-prediction")

for name, param_grid in param_grids.items():
    print(f"\nTuning {name}...")
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("model", param_grid["model"][0])
    ])
    grid = {k: v for k, v in param_grid.items() if k != "model"}
    search = GridSearchCV(pipe, grid, scoring="roc_auc",
                          cv=cv, n_jobs=-1, verbose=0)
    search.fit(X_train, y_train)

    y_pred = search.best_estimator_.predict(X_test)
    y_proba = search.best_estimator_.predict_proba(X_test)[:, 1]

    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)
    f1 = f1_score(y_test, y_pred)

    with mlflow.start_run(run_name=f"{name}_churn"):
        mlflow.log_params(search.best_params_)
        mlflow.log_metric("accuracy", acc)
        mlflow.log_metric("auc", auc)
        mlflow.log_metric("f1", f1)
        mlflow.sklearn.log_model(search.best_estimator_, "model")

    best_models[name] = search.best_estimator_
    results[name] = {"accuracy": acc, "auc": auc, "f1": f1}
    print(f"  Accuracy: {acc:.3f} | AUC: {auc:.3f} | F1: {f1:.3f}")

2026/06/14 18:05:38 INFO mlflow.tracking.fluent: Experiment with name 'churn-prediction' does not exist. Creating a new experiment.



Tuning LogisticRegression...
  Accuracy: 0.797 | AUC: 0.841 | F1: 0.584

Tuning RandomForest...
  Accuracy: 0.753 | AUC: 0.840 | F1: 0.628

Tuning XGBoost...
  Accuracy: 0.742 | AUC: 0.842 | F1: 0.626


In [15]:
print("=" * 55)
print(f"{'Modèle':<22} {'Accuracy':>9} {'AUC':>7} {'F1':>7}")
print("=" * 55)
for name, r in results.items():
    print(f"{name:<22} {r['accuracy']:>9.3f} {r['auc']:>7.3f} {r['f1']:>7.3f}")
print("=" * 55)

best_name = max(results, key=lambda x: results[x]['auc'])
best_model = best_models[best_name]

print(f"\nMeilleur modèle : {best_name}")
print(f"AUC : {results[best_name]['auc']:.3f}")

# Sauvegarder
Path("../model").mkdir(exist_ok=True)
joblib.dump(best_model, "../model/model.pkl")

# Sauvegarder aussi les noms des colonnes pour l'inférence
feature_names = list(X.columns)
joblib.dump(feature_names, "../model/feature_names.pkl")

print(f"Modèle sauvegardé : model/model.pkl")
print(f"Features sauvegardées : model/feature_names.pkl ({len(feature_names)} features)")

print("\n=== Rapport final ===")
print(classification_report(y_test, best_model.predict(X_test),
      target_names=["No Churn", "Churn"]))

Modèle                  Accuracy     AUC      F1
LogisticRegression         0.797   0.841   0.584
RandomForest               0.753   0.840   0.628
XGBoost                    0.742   0.842   0.626

Meilleur modèle : XGBoost
AUC : 0.842
Modèle sauvegardé : model/model.pkl
Features sauvegardées : model/feature_names.pkl (35 features)

=== Rapport final ===
              precision    recall  f1-score   support

    No Churn       0.91      0.72      0.80      1033
       Churn       0.51      0.81      0.63       374

    accuracy                           0.74      1407
   macro avg       0.71      0.76      0.71      1407
weighted avg       0.81      0.74      0.76      1407



In [12]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import precision_recall_curve

# Scaler propre sur les données originales (sans SMOTE)
scaler_clean = StandardScaler()
X_train_clean = scaler_clean.fit_transform(X_train)
X_test_clean = scaler_clean.transform(X_test)

# 4 modèles avec class_weight natif
models_clean = {
    "LR_weighted": LogisticRegression(
        C=0.1, penalty='l2', solver='liblinear',
        class_weight='balanced', max_iter=2000, random_state=42
    ),
    "RF_weighted": RandomForestClassifier(
        n_estimators=300, max_depth=5, min_samples_split=2,
        class_weight='balanced', random_state=42
    ),
    "XGB_weighted": XGBClassifier(
        n_estimators=200, max_depth=4, learning_rate=0.1,
        subsample=0.8, colsample_bytree=0.8,
        scale_pos_weight=(y_train==0).sum()/(y_train==1).sum(),
        random_state=42, eval_metric='logloss'
    ),
    "GradientBoosting": GradientBoostingClassifier(
        n_estimators=200, max_depth=4, learning_rate=0.05,
        subsample=0.8, random_state=42
    )
}

results_clean = {}
trained_clean = {}

mlflow.set_experiment("churn-clean-approach")

for name, model in models_clean.items():
    model.fit(X_train_clean, y_train)
    y_proba = model.predict_proba(X_test_clean)[:, 1]

    # Seuil optimal via courbe Precision-Recall
    precisions, recalls, thresholds = precision_recall_curve(y_test, y_proba)
    f1_scores = 2 * precisions * recalls / (precisions + recalls + 1e-8)
    best_idx = np.argmax(f1_scores)
    best_thresh = thresholds[best_idx]
    y_pred_opt = (y_proba >= best_thresh).astype(int)

    acc = accuracy_score(y_test, y_pred_opt)
    auc = roc_auc_score(y_test, y_proba)
    f1 = f1_score(y_test, y_pred_opt)

    with mlflow.start_run(run_name=f"{name}_clean"):
        mlflow.log_metric("accuracy", acc)
        mlflow.log_metric("auc", auc)
        mlflow.log_metric("f1", f1)
        mlflow.log_param("threshold", round(best_thresh, 3))

    results_clean[name] = {
        "accuracy": acc, "auc": auc,
        "f1": f1, "threshold": best_thresh
    }
    trained_clean[name] = (model, best_thresh)

    print(f"{name:<20} | Seuil: {best_thresh:.3f} | "
          f"Accuracy: {acc:.3f} | AUC: {auc:.3f} | F1: {f1:.3f}")

2026/06/14 18:12:47 INFO mlflow.tracking.fluent: Experiment with name 'churn-clean-approach' does not exist. Creating a new experiment.


LR_weighted          | Seuil: 0.596 | Accuracy: 0.777 | AUC: 0.840 | F1: 0.636
RF_weighted          | Seuil: 0.512 | Accuracy: 0.756 | AUC: 0.838 | F1: 0.636
XGB_weighted         | Seuil: 0.562 | Accuracy: 0.769 | AUC: 0.828 | F1: 0.623
GradientBoosting     | Seuil: 0.311 | Accuracy: 0.765 | AUC: 0.836 | F1: 0.629


In [13]:
voting_clean = VotingClassifier(
    estimators=[
        ('lr',  models_clean['LR_weighted']),
        ('rf',  models_clean['RF_weighted']),
        ('xgb', models_clean['XGB_weighted']),
        ('gb',  models_clean['GradientBoosting'])
    ],
    voting='soft'
)
voting_clean.fit(X_train_clean, y_train)
y_proba_vc = voting_clean.predict_proba(X_test_clean)[:, 1]

precisions, recalls, thresholds = precision_recall_curve(y_test, y_proba_vc)
f1_scores = 2 * precisions * recalls / (precisions + recalls + 1e-8)
best_thresh_vc = thresholds[np.argmax(f1_scores)]
y_pred_vc = (y_proba_vc >= best_thresh_vc).astype(int)

acc_vc = accuracy_score(y_test, y_pred_vc)
auc_vc = roc_auc_score(y_test, y_proba_vc)
f1_vc  = f1_score(y_test, y_pred_vc)

results_clean["VotingClassifier"] = {
    "accuracy": acc_vc, "auc": auc_vc,
    "f1": f1_vc, "threshold": best_thresh_vc
}
trained_clean["VotingClassifier"] = (voting_clean, best_thresh_vc)

print(f"{'VotingClassifier':<20} | Seuil: {best_thresh_vc:.3f} | "
      f"Accuracy: {acc_vc:.3f} | AUC: {auc_vc:.3f} | F1: {f1_vc:.3f}")

VotingClassifier     | Seuil: 0.560 | Accuracy: 0.790 | AUC: 0.840 | F1: 0.632


In [14]:
print("\n" + "=" * 65)
print(f"{'Modèle':<22} {'AUC':>7} {'F1':>7} {'Seuil':>7} {'Accuracy':>9}")
print("=" * 65)

best_f1_name = max(results_clean, key=lambda x: results_clean[x]['f1'])

for name, r in results_clean.items():
    marker = " ◄ meilleur" if name == best_f1_name else ""
    print(f"{name:<22} {r['auc']:>7.3f} {r['f1']:>7.3f} "
          f"{r['threshold']:>7.3f} {r['accuracy']:>9.3f}{marker}")

print("=" * 65)

# Comparaison avec baseline
print(f"\nBaseline F1  : 0.626")
print(f"Meilleur F1  : {results_clean[best_f1_name]['f1']:.3f}")
print(f"Gain F1      : +{results_clean[best_f1_name]['f1'] - 0.626:.3f}")

# Sauvegarder le meilleur
best_model_c, best_thresh_c = trained_clean[best_f1_name]

joblib.dump(best_model_c,  "../model/model.pkl")
joblib.dump(scaler_clean,  "../model/scaler.pkl")
joblib.dump(best_thresh_c, "../model/threshold.pkl")
joblib.dump(list(X.columns), "../model/feature_names.pkl")

print(f"\nModèle sauvegardé : {best_f1_name}")
print(f"AUC final : {results_clean[best_f1_name]['auc']:.3f}")
print(f"F1 final  : {results_clean[best_f1_name]['f1']:.3f}")

print("\n=== Rapport de classification ===")
print(classification_report(
    y_test, y_pred_vc,
    target_names=["No Churn", "Churn"]
))


Modèle                     AUC      F1   Seuil  Accuracy
LR_weighted              0.840   0.636   0.596     0.777 ◄ meilleur
RF_weighted              0.838   0.636   0.512     0.756
XGB_weighted             0.828   0.623   0.562     0.769
GradientBoosting         0.836   0.629   0.311     0.765
VotingClassifier         0.840   0.632   0.560     0.790

Baseline F1  : 0.626
Meilleur F1  : 0.636
Gain F1      : +0.010

Modèle sauvegardé : LR_weighted
AUC final : 0.840
F1 final  : 0.636

=== Rapport de classification ===
              precision    recall  f1-score   support

    No Churn       0.88      0.83      0.85      1033
       Churn       0.59      0.68      0.63       374

    accuracy                           0.79      1407
   macro avg       0.73      0.75      0.74      1407
weighted avg       0.80      0.79      0.79      1407

